In [3]:
# Šioje celėje importuojamos visos reikalingos bibliotekos.
# Jos naudojamos duomenų skaitymui, teksto apdorojimui, Lead-3 santraukų generavimui,
# metrikų skaičiavimui ir grafikų braižymui.

#Lead-3 metodas yra paprastas ekstraktyvinis santraukų generavimo metodas, kuris nesiremia mokymu ir neturi treniruojamų parametrų. Jis naudojamas kaip bazinis (baseline) metodas, leidžiantis įvertinti, ar sudėtingesni modeliai iš tiesų suteikia reikšmingą pagerėjimą.
import json
import re
from dataclasses import asdict, dataclass
from pathlib import Path
import torch

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from rouge_score import rouge_scorer
from bert_score import score as bertscore_score

In [4]:
# Šioje celėje aprašoma konfigūracija, t. y. visi pagrindiniai nustatymai vienoje vietoje.
# Čia nurodomi duomenų failų keliai, stulpelių pavadinimai, išvesties aplankas,
# Lead-3 metodo parametrai ir vertinimo metrikų nustatymai.

@dataclass
class Config:
    train_csv: str = r"C:\\bakis\\duomenys\\samsum-train.csv"
    val_csv: str = r"C:\\bakis\\duomenys\\samsum-validation.csv"
    test_csv: str = r"C:\\bakis\\duomenys\\samsum-test.csv"

    text_column: str = "dialogue"
    summary_column: str = "summary"

    output_dir: str = r"C:\\bakis\\PALYGINIMAS\\lead3"

    # Naudosime tik dalį train, kad palyginimas tarp modelių būtų greitesnis
    use_train_subset: bool = True
    train_subset_size: int = 5000
    subset_random_state: int = 42

    # Lead-3
    max_sentences: int = 3
    fallback_to_full_text_if_no_sentence_split: bool = True

    bertscore_model_type: str = "bert-base-uncased"
    bertscore_lang: str = "en"
    compute_bertscore_on_test: bool = True
    bertscore_batch_size: int = 16

    
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


    # Output
    save_predictions_csv: bool = True
    make_plots: bool = True
    dpi: int = 160


cfg = Config()
cfg

Config(train_csv='C:\\\\bakis\\\\duomenys\\\\samsum-train.csv', val_csv='C:\\\\bakis\\\\duomenys\\\\samsum-validation.csv', test_csv='C:\\\\bakis\\\\duomenys\\\\samsum-test.csv', text_column='dialogue', summary_column='summary', output_dir='C:\\\\bakis\\\\PALYGINIMAS\\\\lead3', use_train_subset=True, train_subset_size=5000, subset_random_state=42, max_sentences=3, fallback_to_full_text_if_no_sentence_split=True, bertscore_model_type='bert-base-uncased', bertscore_lang='en', compute_bertscore_on_test=True, bertscore_batch_size=16, device='cuda', save_predictions_csv=True, make_plots=True, dpi=160)

In [5]:
# Šioje celėje aprašomos pagalbinės funkcijos, kurios bus naudojamos visame notebook'e.
# Jos sukuria aplankus, sutvarko tekstą, suskaido tekstą į sakinius,
# sugeneruoja Lead-3 santrauką ir apskaičiuoja žodžių kiekį.

def ensure_dir(path):
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p


def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text


def sentence_split(text):
    text = normalize_text(text)
    if not text:
        return []
    parts = re.split(r"(?<=[.!?])\s+", text)
    parts = [p.strip() for p in parts if p.strip()]
    return parts


def lead3_summarize(text, max_sentences=3, fallback_to_full_text_if_no_sentence_split=True):
    text = normalize_text(text)
    if not text:
        return ""

    sents = sentence_split(text)

    if len(sents) == 0:
        return text if fallback_to_full_text_if_no_sentence_split else ""

    return " ".join(sents[:max_sentences]).strip()


def word_count(text):
    text = normalize_text(text)
    if not text:
        return 0
    return len(text.split())


def safe_mean(values):
    return float(np.mean(values)) if len(values) > 0 else 0.0

In [6]:
# Šioje celėje aprašomos duomenų įkėlimo funkcijos.
# Jos perskaito CSV failą, patikrina, ar yra reikalingi stulpeliai,
# pašalina tuščius įrašus ir, jei reikia, paima tik dalį train duomenų.

def load_single_dataset(csv_path, cfg):
    df = pd.read_csv(csv_path)

    if cfg.text_column not in df.columns:
        raise ValueError(f"Nerastas teksto stulpelis: {cfg.text_column}")
    if cfg.summary_column not in df.columns:
        raise ValueError(f"Nerastas santraukos stulpelis: {cfg.summary_column}")

    df = df[[cfg.text_column, cfg.summary_column]].copy()
    df[cfg.text_column] = df[cfg.text_column].apply(normalize_text)
    df[cfg.summary_column] = df[cfg.summary_column].apply(normalize_text)

    df = df[
        (df[cfg.text_column] != "") &
        (df[cfg.summary_column] != "")
    ].reset_index(drop=True)

    return df


def maybe_take_train_subset(df, cfg):
    if not cfg.use_train_subset:
        return df.reset_index(drop=True)

    subset_size = min(cfg.train_subset_size, len(df))
    return df.sample(n=subset_size, random_state=cfg.subset_random_state).reset_index(drop=True)

In [7]:
# Šioje celėje įkeliami train, validation ir test duomenų rinkiniai.
# Train rinkiniui gali būti pritaikomas sumažinimas, kad eksperimentas vyktų greičiau.
# Pabaigoje atspausdinama, kiek įrašų yra kiekviename rinkinyje.

full_train_df = load_single_dataset(cfg.train_csv, cfg)
val_df = load_single_dataset(cfg.val_csv, cfg)
test_df = load_single_dataset(cfg.test_csv, cfg)

train_df = maybe_take_train_subset(full_train_df, cfg)

print("Full train:", len(full_train_df))
print("Used train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Full train: 14731
Used train: 5000
Validation: 818
Test: 819


In [8]:
# Šioje celėje apskaičiuojama ir atspausdinama pagrindinė duomenų statistika.
# Rodomas pavyzdžių skaičius bei vidutinis dialogo ir santraukos ilgis žodžiais.
# Tai padeda suprasti, kokio dydžio yra tekstai, su kuriais dirba modelis.

def print_dataset_stats(df, cfg, name):
    source_lengths = df[cfg.text_column].apply(word_count)
    summary_lengths = df[cfg.summary_column].apply(word_count)

    print(f"\n{name}")
    print("-" * len(name))
    print(f"Samples: {len(df)}")
    print(f"Avg source words:  {source_lengths.mean():.2f}")
    print(f"Avg summary words: {summary_lengths.mean():.2f}")


print_dataset_stats(train_df, cfg, "TRAIN")
print_dataset_stats(val_df, cfg, "VALIDATION")
print_dataset_stats(test_df, cfg, "TEST")


TRAIN
-----
Samples: 5000
Avg source words:  95.82
Avg summary words: 20.56

VALIDATION
----------
Samples: 818
Avg source words:  91.64
Avg summary words: 20.28

TEST
----
Samples: 819
Avg source words:  95.51
Avg summary words: 20.02


In [9]:
# Šioje celėje sukuriama funkcija, kuri kiekvienam duomenų rinkinio įrašui sugeneruoja prognozę.
# Prognozė gaunama taikant Lead-3 metodą: paimami pirmi keli dialogo sakiniai.
# Rezultatas saugomas lentelėje kartu su originaliu tekstu, tikra santrauka ir žodžių skaičiais.

def generate_predictions(df, cfg, split_name):
    rows = []

    for idx, row in df.iterrows():
        source_text = row[cfg.text_column]
        reference = row[cfg.summary_column]

        prediction = lead3_summarize(
            source_text,
            max_sentences=cfg.max_sentences,
            fallback_to_full_text_if_no_sentence_split=cfg.fallback_to_full_text_if_no_sentence_split
        )

        rows.append({
        "split": split_name,
        "id": len(rows),
        "source_text": source_text,
        "reference_summary": reference,
        "predicted_summary": prediction,
        "source_words": word_count(source_text),
        "reference_words": word_count(reference),
        "prediction_words": word_count(prediction),
        })

    return pd.DataFrame(rows)

In [10]:
# Šioje celėje skaičiuojamos ROUGE metrikos.
# ROUGE lygina modelio sugeneruotą santrauką su tikrąja santrauka pagal žodžių sutapimus.
# Apskaičiuojamos ROUGE-1, ROUGE-2 ir ROUGE-L precision, recall bei F1 reikšmės.

def compute_rouge(pred_df):
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

    rouge1_p, rouge1_r, rouge1_f = [], [], []
    rouge2_p, rouge2_r, rouge2_f = [], [], []
    rougeL_p, rougeL_r, rougeL_f = [], [], []

    for _, row in pred_df.iterrows():
        reference = row["reference_summary"]
        prediction = row["predicted_summary"]

        scores = scorer.score(reference, prediction)

        r1 = scores["rouge1"]
        r2 = scores["rouge2"]
        rl = scores["rougeL"]

        rouge1_p.append(r1.precision)
        rouge1_r.append(r1.recall)
        rouge1_f.append(r1.fmeasure)

        rouge2_p.append(r2.precision)
        rouge2_r.append(r2.recall)
        rouge2_f.append(r2.fmeasure)

        rougeL_p.append(rl.precision)
        rougeL_r.append(rl.recall)
        rougeL_f.append(rl.fmeasure)

    pred_df = pred_df.copy()
    pred_df["rouge1_precision"] = rouge1_p
    pred_df["rouge1_recall"] = rouge1_r
    pred_df["rouge1_f1"] = rouge1_f

    pred_df["rouge2_precision"] = rouge2_p
    pred_df["rouge2_recall"] = rouge2_r
    pred_df["rouge2_f1"] = rouge2_f

    pred_df["rougeL_precision"] = rougeL_p
    pred_df["rougeL_recall"] = rougeL_r
    pred_df["rougeL_f1"] = rougeL_f

    return pred_df

In [11]:
# Šioje celėje skaičiuojama BERTScore metrika.
# BERTScore vertina ne tik tikslius žodžių sutapimus, bet ir semantinį panašumą tarp santraukų.
# Funkcija prie rezultatų lentelės prideda precision, recall ir F1 reikšmes.

def compute_bertscore(pred_df, cfg):
    refs = pred_df["reference_summary"].fillna("").astype(str).tolist()
    preds = pred_df["predicted_summary"].fillna("").astype(str).tolist()

    P, R, F1 = bertscore_score(
        preds,
        refs,
        lang=cfg.bertscore_lang,
        model_type=cfg.bertscore_model_type,
        batch_size=cfg.bertscore_batch_size,
        verbose=True,
        device=cfg.device
    )

    pred_df = pred_df.copy()
    pred_df["bertscore_precision"] = P.cpu().numpy()
    pred_df["bertscore_recall"] = R.cpu().numpy()
    pred_df["bertscore_f1"] = F1.cpu().numpy()

    return pred_df

In [12]:
# Šioje celėje aprašoma vieno duomenų rinkinio vertinimo eiga.
# Pirmiausia sugeneruojamos Lead-3 santraukos, tada apskaičiuojamos ROUGE ir BERTScore metrikos.
# Galiausiai grąžinama rezultatų lentelė ir bendros vidutinės metrikos.

def evaluate_split(df, cfg, split_name):
    pred_df = generate_predictions(df, cfg, split_name)
    pred_df = compute_rouge(pred_df)
    pred_df = compute_bertscore(pred_df, cfg)

    metrics = {
        "num_samples": int(len(pred_df)),
        "rouge1_f1_mean": safe_mean(pred_df["rouge1_f1"].values),
        "rouge2_f1_mean": safe_mean(pred_df["rouge2_f1"].values),
        "rougeL_f1_mean": safe_mean(pred_df["rougeL_f1"].values),
        "bertscore_f1_mean": safe_mean(pred_df["bertscore_f1"].values),

        "rouge1_precision_mean": safe_mean(pred_df["rouge1_precision"].values),
        "rouge1_recall_mean": safe_mean(pred_df["rouge1_recall"].values),

        "rouge2_precision_mean": safe_mean(pred_df["rouge2_precision"].values),
        "rouge2_recall_mean": safe_mean(pred_df["rouge2_recall"].values),

        "rougeL_precision_mean": safe_mean(pred_df["rougeL_precision"].values),
        "rougeL_recall_mean": safe_mean(pred_df["rougeL_recall"].values),

        "bertscore_precision_mean": safe_mean(pred_df["bertscore_precision"].values),
        "bertscore_recall_mean": safe_mean(pred_df["bertscore_recall"].values),

        "source_words_mean": safe_mean(pred_df["source_words"].values),
        "reference_words_mean": safe_mean(pred_df["reference_words"].values),
        "prediction_words_mean": safe_mean(pred_df["prediction_words"].values),
    }

    return pred_df, metrics

In [13]:
# Šioje celėje aprašoma funkcija, kuri aiškiai išveda metrikas į ekraną.
# Ji naudojama tam, kad train, validation ir test rezultatų reikšmės būtų lengvai perskaitomos.

def print_metrics(metrics, split_name):
    print(f"\n===== {split_name.upper()} METRICS =====")
    print(f"Samples:               {metrics['num_samples']}")
    print(f"ROUGE-1 F1:            {metrics['rouge1_f1_mean']:.4f}")
    print(f"ROUGE-2 F1:            {metrics['rouge2_f1_mean']:.4f}")
    print(f"ROUGE-L F1:            {metrics['rougeL_f1_mean']:.4f}")
    print(f"BERTScore F1:          {metrics['bertscore_f1_mean']:.4f}")
    print(f"ROUGE-1 Precision:     {metrics['rouge1_precision_mean']:.4f}")
    print(f"ROUGE-1 Recall:        {metrics['rouge1_recall_mean']:.4f}")
    print(f"ROUGE-2 Precision:     {metrics['rouge2_precision_mean']:.4f}")
    print(f"ROUGE-2 Recall:        {metrics['rouge2_recall_mean']:.4f}")
    print(f"ROUGE-L Precision:     {metrics['rougeL_precision_mean']:.4f}")
    print(f"ROUGE-L Recall:        {metrics['rougeL_recall_mean']:.4f}")
    print(f"BERTScore Precision:   {metrics['bertscore_precision_mean']:.4f}")
    print(f"BERTScore Recall:      {metrics['bertscore_recall_mean']:.4f}")
    print(f"Avg source words:      {metrics['source_words_mean']:.2f}")
    print(f"Avg reference words:   {metrics['reference_words_mean']:.2f}")
    print(f"Avg prediction words:  {metrics['prediction_words_mean']:.2f}")

In [14]:
# Šioje celėje aprašomos funkcijos rezultatams išsaugoti į failus.
# Prognozės gali būti saugomos CSV arba JSON formatu, o metrikos išsaugomos JSON faile.
# Tai leidžia vėliau palyginti šį baseline metodą su kitais modeliais.

def save_dataframe(df, path):
    path = Path(path)
    if path.suffix.lower() == ".csv":
        df.to_csv(path, index=False, encoding="utf-8")
    elif path.suffix.lower() == ".json":
        df.to_json(path, orient="records", force_ascii=False, indent=2)
    else:
        raise ValueError(f"Nepalaikomas failo tipas: {path}")


def save_metrics(metrics, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

In [15]:
# Šioje celėje išsaugoma Lead-3 baseline metodo informacija.
# Kadangi Lead-3 neturi treniruojamų svorių, saugomi tik metodo nustatymai ir aprašymas.
# Taip užtikrinama, kad eksperimentą būtų galima pakartoti tomis pačiomis sąlygomis.

def save_baseline_artifacts(cfg, output_dir):
    output_dir = ensure_dir(output_dir)
    model_dir = ensure_dir(output_dir / "model")
    splits_dir = ensure_dir(output_dir / "splits")

    model_artifact = {
        "model_name": "lead3_baseline",
        "model_type": "extractive_baseline",
        "description": "Returns the first N sentences from the source text.",
        "parameters": {
            "max_sentences": cfg.max_sentences,
            "fallback_to_full_text_if_no_sentence_split": cfg.fallback_to_full_text_if_no_sentence_split,
        },
        "notes": [
            "This model is rule-based and has no trainable weights.",
            "Saved for reproducibility.",
            "Train subset may be used for fairer comparison with heavier models."
        ]
    }

    with open(model_dir / "model_config.json", "w", encoding="utf-8") as f:
        json.dump(model_artifact, f, ensure_ascii=False, indent=2)

    with open(model_dir / "experiment_config.json", "w", encoding="utf-8") as f:
        json.dump(asdict(cfg), f, ensure_ascii=False, indent=2)

    training_history = {
        "model_name": "lead3_baseline",
        "trained": False,
        "epochs": 0,
        "train_loss": [],
        "val_loss": [],
        "notes": [
            "Lead-3 baseline does not require training.",
            "No optimization history exists."
        ]
    }

    with open(output_dir / "training_history.json", "w", encoding="utf-8") as f:
        json.dump(training_history, f, ensure_ascii=False, indent=2)

    dataset_info = {
        "full_train_size": int(len(full_train_df)),
        "used_train_size": int(len(train_df)),
        "validation_size": int(len(val_df)),
        "test_size": int(len(test_df)),
        "used_train_subset": bool(cfg.use_train_subset),
        "train_subset_size_requested": int(cfg.train_subset_size),
    }

    with open(output_dir / "dataset_info.json", "w", encoding="utf-8") as f:
        json.dump(dataset_info, f, ensure_ascii=False, indent=2)

    save_dataframe(train_df, splits_dir / "train_used.csv")
    save_dataframe(val_df, splits_dir / "val.csv")
    save_dataframe(test_df, splits_dir / "test.csv")

In [16]:
# Šioje celėje iš rezultatų lentelės sudaromas bendras metrikų žodynas.
# Apskaičiuojamos vidutinės ROUGE, BERTScore ir teksto ilgio reikšmės.
# Šis žodynas vėliau naudojamas rezultatų išsaugojimui ir grafikų braižymui.

def build_metrics(pred_df):
    metrics = {
        "num_samples": len(pred_df),

        "rouge1_f1_mean": float(pred_df["rouge1_f1"].mean()),
        "rouge2_f1_mean": float(pred_df["rouge2_f1"].mean()),
        "rougeL_f1_mean": float(pred_df["rougeL_f1"].mean()),

        "rouge1_precision_mean": float(pred_df["rouge1_precision"].mean()),
        "rouge1_recall_mean": float(pred_df["rouge1_recall"].mean()),

        "rouge2_precision_mean": float(pred_df["rouge2_precision"].mean()),
        "rouge2_recall_mean": float(pred_df["rouge2_recall"].mean()),

        "rougeL_precision_mean": float(pred_df["rougeL_precision"].mean()),
        "rougeL_recall_mean": float(pred_df["rougeL_recall"].mean()),

        "source_words_mean": float(pred_df["source_words"].mean()),
        "reference_words_mean": float(pred_df["reference_words"].mean()),
        "prediction_words_mean": float(pred_df["prediction_words"].mean()),
    }

    if "bertscore_precision" in pred_df.columns:
        metrics["bertscore_precision_mean"] = float(pred_df["bertscore_precision"].mean())
        metrics["bertscore_recall_mean"] = float(pred_df["bertscore_recall"].mean())
        metrics["bertscore_f1_mean"] = float(pred_df["bertscore_f1"].mean())

    return metrics

In [17]:
# Šioje celėje aprašomos pagalbinės grafikų braižymo funkcijos.
# Jos sukuria stulpelines diagramas, histogramas ir taškines diagramas.
# Grafikai padeda vizualiai įvertinti modelio rezultatų pasiskirstymą.

def save_bar_plot(metric_names, metric_values, output_path, title, dpi=160):
    plt.figure(figsize=(8, 5))
    plt.bar(metric_names, metric_values)
    plt.title(title)
    plt.ylabel("Value")
    plt.xlabel("Metric")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(output_path, dpi=dpi, bbox_inches="tight")
    plt.show()
    plt.close()


def save_histogram(values, output_path, title, xlabel, dpi=160):
    plt.figure(figsize=(8, 5))
    plt.hist(pd.Series(values).dropna(), bins=30)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(output_path, dpi=dpi, bbox_inches="tight")
    plt.show()
    plt.close()


def save_scatter(x, y, output_path, title, xlabel, ylabel, dpi=160):
    plt.figure(figsize=(7, 5))
    plt.scatter(x, y, alpha=0.5)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(output_path, dpi=dpi, bbox_inches="tight")
    plt.show()
    plt.close()

In [18]:
# Šioje celėje aprašoma funkcija, kuri sugeneruoja visus pagrindinius grafikus vienam duomenų rinkiniui.
# Ji nubraižo bendrų metrikų palyginimą, atskirų metrikų histogramas ir ryšius tarp ilgių bei kokybės metrikų.
# Visi grafikai išsaugomi atitinkamame rezultatų aplanke.

def make_plots(pred_df, metrics, split_dir, cfg):
    plots_dir = ensure_dir(split_dir / "plots")

    save_bar_plot(
        ["ROUGE-1 F1", "ROUGE-2 F1", "ROUGE-L F1", "BERTScore F1"],
        [
            metrics["rouge1_f1_mean"],
            metrics["rouge2_f1_mean"],
            metrics["rougeL_f1_mean"],
            metrics["bertscore_f1_mean"]
        ],
        plots_dir / "main_metrics_bar.png", 
        f"{split_dir.name}: main metrics",
        dpi=cfg.dpi
    )

    save_histogram(
        pred_df["rouge1_f1"],
        plots_dir / "rouge1_f1_hist.png",
        f"{split_dir.name}: ROUGE-1 F1 distribution",
        "ROUGE-1 F1",
        dpi=cfg.dpi
    )

    save_histogram(
        pred_df["rouge2_f1"],
        plots_dir / "rouge2_f1_hist.png",
        f"{split_dir.name}: ROUGE-2 F1 distribution",
        "ROUGE-2 F1",
        dpi=cfg.dpi
    )

    save_histogram(
        pred_df["rougeL_f1"],
        plots_dir / "rougeL_f1_hist.png",
        f"{split_dir.name}: ROUGE-L F1 distribution",
        "ROUGE-L F1",
        dpi=cfg.dpi
    )

    save_histogram(
        pred_df["bertscore_f1"],
        plots_dir / "bertscore_f1_hist.png",
        f"{split_dir.name}: BERTScore F1 distribution",
        "BERTScore F1",
        dpi=cfg.dpi
    )

    save_histogram(
        pred_df["prediction_words"],
        plots_dir / "prediction_length_hist.png",
        f"{split_dir.name}: prediction length distribution",
        "Prediction words",
        dpi=cfg.dpi
    )

    save_scatter(
        pred_df["source_words"],
        pred_df["rouge1_f1"],
        plots_dir / "source_len_vs_rouge1_f1.png",
        f"{split_dir.name}: source length vs ROUGE-1 F1",
        "Source words",
        "ROUGE-1 F1",
        dpi=cfg.dpi
    )

    save_scatter(
        pred_df["prediction_words"],
        pred_df["bertscore_f1"],
        plots_dir / "prediction_len_vs_bertscore_f1.png",
        f"{split_dir.name}: prediction length vs BERTScore F1",
        "Prediction words",
        "BERTScore F1",
        dpi=cfg.dpi
    )

In [19]:
# Šioje celėje vykdomas pagrindinis eksperimentas.
# Kiekvienam duomenų rinkiniui sugeneruojamos Lead-3 santraukos, apskaičiuojamos metrikos,
# išsaugomi rezultatai ir sukuriami grafikai. Pabaigoje bendri rezultatai išsaugomi į summary failą.

output_dir = ensure_dir(cfg.output_dir)
save_baseline_artifacts(cfg, output_dir)

all_results = {}

split_map = {
    "train": train_df,
    "val": val_df,
    "test": test_df,
}

for split_name, split_df in split_map.items():
    print(f"\nProcessing {split_name}...")

    split_dir = ensure_dir(output_dir / split_name)

    pred_df = generate_predictions(split_df, cfg, split_name)
    pred_df = compute_rouge(pred_df)

    if split_name == "test" and cfg.compute_bertscore_on_test:
        pred_df = compute_bertscore(pred_df, cfg) 

    metrics = build_metrics(pred_df)
    all_results[split_name] = metrics

    pred_df.to_csv(split_dir / "predictions.csv", index=False, encoding="utf-8")

    with open(split_dir / "metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

    print(metrics)

with open(output_dir / "all_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print("\nDone.")


Processing train...
{'num_samples': 5000, 'rouge1_f1_mean': 0.31282935952377533, 'rouge2_f1_mean': 0.0964399967746851, 'rougeL_f1_mean': 0.24098768973556722, 'rouge1_precision_mean': 0.28213949662614896, 'rouge1_recall_mean': 0.4539562237541166, 'rouge2_precision_mean': 0.08394351020453876, 'rouge2_recall_mean': 0.14572177008941598, 'rougeL_precision_mean': 0.2173522891308514, 'rougeL_recall_mean': 0.35002111985315143, 'source_words_mean': 95.82, 'reference_words_mean': 20.5566, 'prediction_words_mean': 34.0094}

Processing val...
{'num_samples': 818, 'rouge1_f1_mean': 0.3150793805149375, 'rouge2_f1_mean': 0.10174639741588597, 'rougeL_f1_mean': 0.24374374757122416, 'rouge1_precision_mean': 0.2876457791983706, 'rouge1_recall_mean': 0.46176189788920236, 'rouge2_precision_mean': 0.08883728074009972, 'rouge2_recall_mean': 0.15763906557921353, 'rougeL_precision_mean': 0.2215286498687115, 'rougeL_recall_mean': 0.35879117780667186, 'source_words_mean': 91.64180929095355, 'reference_words_mea

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9897.15it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


100%|██████████| 103/103 [00:02<00:00, 34.44it/s]


computing greedy matching.


100%|██████████| 52/52 [00:00<00:00, 156.26it/s]

done in 3.33 seconds, 245.82 sentences/sec
{'num_samples': 819, 'rouge1_f1_mean': 0.307525246319011, 'rouge2_f1_mean': 0.08923874243854206, 'rougeL_f1_mean': 0.2353729805378047, 'rouge1_precision_mean': 0.27782374673744, 'rouge1_recall_mean': 0.45075501688538594, 'rouge2_precision_mean': 0.07847755565744226, 'rouge2_recall_mean': 0.13570476281865515, 'rougeL_precision_mean': 0.21318809105839576, 'rougeL_recall_mean': 0.34560545412573745, 'source_words_mean': 95.5079365079365, 'reference_words_mean': 20.017094017094017, 'prediction_words_mean': 34.14041514041514, 'bertscore_precision_mean': 0.44781285524368286, 'bertscore_recall_mean': 0.5546672940254211, 'bertscore_f1_mean': 0.4914059042930603}

Done.


In [21]:
# Šioje celėje peržiūrimi keli test rinkinio prognozių pavyzdžiai.
# Rodomas originalus dialogas, tikroji santrauka, Lead-3 sugeneruota santrauka ir kelios metrikos.
# Tai leidžia ne tik skaičiais, bet ir vizualiai įvertinti, kaip veikia metodas.

test_predictions = pd.read_csv(Path(cfg.output_dir) / "test" / "predictions.csv")
test_predictions[["source_text", "reference_summary", "predicted_summary", "rouge1_f1", "rougeL_f1", "bertscore_f1"]].head(10)

,source_text,reference_summary,predicted_summary,rouge1_f1,rougeL_f1,bertscore_f1
0,"Hannah: Hey, do you have Betty's number? Amand...",Hannah needs Betty's number but Amanda doesn't...,"Hannah: Hey, do you have Betty's number? Amand...",0.240000,0.213333,0.491240
1,Eric: MACHINE! Rob: That's so gr8! Eric: I kno...,Eric and Rob are going to watch a stand-up on ...,Eric: MACHINE! Rob: That's so gr8! Eric: I know!,0.181818,0.181818,0.361173
2,"Lenny: Babe, can you help me with something? B...",Lenny can't decide which trousers to buy. Bob ...,"Lenny: Babe, can you help me with something? B...",0.333333,0.250000,0.438862
3,"Will: hey babe, what do you want for dinner to...",Emma will be home soon and she will let Will k...,"Will: hey babe, what do you want for dinner to...",0.162162,0.108108,0.333620
4,"Ollie: Hi , are you in Warsaw Jane: yes, just ...",Jane is in Warsaw. Ollie and Jane has a party....,"Ollie: Hi , are you in Warsaw Jane: yes, just ...",0.190476,0.126984,0.466774
5,"Benjamin: Hey guys, what are we doing with the...",Hilary has the keys to the apartment. Benjamin...,"Benjamin: Hey guys, what are we doing with the...",0.252874,0.160920,0.497067
6,Max: Know any good sites to buy clothes from? ...,Payton provides Max with websites selling clot...,Max: Know any good sites to buy clothes from? ...,0.225000,0.175000,0.436537
7,Rita: I'm so bloody tired. Falling asleep at w...,Rita and Tina are bored at work and have still...,Rita: I'm so bloody tired. Falling asleep at w...,0.275862,0.206897,0.448852
8,"Beatrice: I am in town, shopping. They have ni...","Beatrice wants to buy Leo a scarf, but he does...","Beatrice: I am in town, shopping. They have ni...",0.163265,0.081633,0.462131
9,Ivan: hey eric Eric: yeah man Ivan: so youre c...,Eric doesn't know if his parents let him go to...,Ivan: hey eric Eric: yeah man Ivan: so youre c...,0.261682,0.168224,0.478740


In [22]:
# Šioje celėje aprašomos funkcijos, leidžiančios naudoti išsaugotą Lead-3 modelio konfigūraciją.
# Pirmoji funkcija įkelia modelio nustatymus iš JSON failo, o antroji pagal juos sugeneruoja santrauką naujam tekstui.

def load_saved_lead3_model(model_dir):
    model_dir = Path(model_dir)
    with open(model_dir / "model_config.json", "r", encoding="utf-8") as f:
        model_cfg = json.load(f)
    return model_cfg


def predict_with_saved_lead3(model_dir, text):
    model_cfg = load_saved_lead3_model(model_dir)
    params = model_cfg["parameters"]

    return lead3_summarize(
        text,
        max_sentences=params["max_sentences"],
        fallback_to_full_text_if_no_sentence_split=params["fallback_to_full_text_if_no_sentence_split"]
    )

In [23]:
# Šioje celėje pateikiamas praktinis pavyzdys, kaip naudoti išsaugotą Lead-3 metodą naujam dialogui.
# Sukuriamas trumpas dialogas, jam sugeneruojama santrauka ir rezultatas atspausdinamas ekrane.

sample_text = """Amanda: Are we still meeting tomorrow?
John: Yes, at 10 in the library.
Amanda: Great, I will bring the notes.
John: Perfect, see you there."""

summary = predict_with_saved_lead3(Path(cfg.output_dir) / "model", sample_text)
print(summary)

Amanda: Are we still meeting tomorrow? John: Yes, at 10 in the library. Amanda: Great, I will bring the notes.
